# Final Logistic Regression with Feature Engineering

This notebook focuses more on **feature engineering** before training a Logistic Regression model.

**Target column:** `income`  
**Problem type:** Binary classification  
**Goal:** Predict whether employee income is `<=50K` or `>50K`.

Feature engineering helps convert raw columns into more meaningful features for machine learning.

In [ ]:
# Import libraries for data handling
import pandas as pd
import numpy as np

# Import visualization library
import matplotlib.pyplot as plt

# Import train-test split
from sklearn.model_selection import train_test_split

# Import preprocessing tools
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Import Logistic Regression model
from sklearn.linear_model import LogisticRegression

# Import model evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay

# Import pickle to save model
import pickle

## 1. Load the dataset

In [ ]:
# Load employee dataset
# Make sure employee.csv is available in the same folder
df = pd.read_csv("employee.csv")

# Display first 5 rows
print(df.head())

## 2. Understand the dataset

In [ ]:
# Check number of rows and columns
print("Dataset shape:", df.shape)

# Check columns and data types
print(df.info())

# Check target column distribution
print(df["income"].value_counts())

## 3. Basic cleaning

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove extra spaces from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Replace '?' with NaN because '?' represents missing or unknown values
df = df.replace("?", np.nan)

# Create missing-value indicator columns before filling values
# These indicators may help the model learn whether missing information has meaning
df["workclass_missing"] = df["workclass"].isna().astype(int)
df["occupation_missing"] = df["occupation"].isna().astype(int)
df["native_country_missing"] = df["native-country"].isna().astype(int)

# Fill missing categorical values with 'Unknown'
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown")

print("Cleaning completed")
print(df.isnull().sum())

## 4. Feature engineering

In this section, we create new useful columns from existing columns.

In [ ]:
# Create age groups from age column
# Age groups can capture life-stage patterns better than raw age alone
def create_age_group(age):
    if age < 25:
        return "Young"
    elif age < 45:
        return "Adult"
    elif age < 60:
        return "Senior"
    else:
        return "Elder"

df["age_group"] = df["age"].apply(create_age_group)

# Create education level from educational-num
# This converts numeric education into broad readable groups
def create_education_level(num):
    if num <= 8:
        return "School_Level"
    elif num <= 12:
        return "High_School_Level"
    elif num <= 14:
        return "Graduate_Level"
    else:
        return "Advanced_Level"

df["education_level"] = df["educational-num"].apply(create_education_level)

# Create marital indicator
# Married people may have different income patterns in this dataset
df["is_married"] = df["marital-status"].apply(lambda x: 1 if "Married" in x else 0)

# Create private workclass indicator
df["is_private_workclass"] = df["workclass"].apply(lambda x: 1 if x == "Private" else 0)

# Create US native-country indicator
df["is_us_native"] = df["native-country"].apply(lambda x: 1 if x == "United-States" else 0)

# Create capital gain and loss indicators
df["has_capital_gain"] = df["capital-gain"].apply(lambda x: 1 if x > 0 else 0)
df["has_capital_loss"] = df["capital-loss"].apply(lambda x: 1 if x > 0 else 0)

# Create net capital amount
df["capital_net"] = df["capital-gain"] - df["capital-loss"]

# Log transform skewed numeric columns
# np.log1p handles zero values safely
df["log_fnlwgt"] = np.log1p(df["fnlwgt"])
df["log_capital_gain"] = np.log1p(df["capital-gain"])
df["log_capital_loss"] = np.log1p(df["capital-loss"])

# Create interaction feature between age and education number
df["age_x_education"] = df["age"] * df["educational-num"]

# Create work intensity category from hours-per-week
def create_work_intensity(hours):
    if hours < 35:
        return "Low_Hours"
    elif hours <= 45:
        return "Standard_Hours"
    else:
        return "High_Hours"

df["work_intensity"] = df["hours-per-week"].apply(create_work_intensity)

print("Feature engineering completed")
print(df.head())

## 5. Visualize important engineered features

In [ ]:
# Check income count by education level
income_by_education = pd.crosstab(df["education_level"], df["income"])

income_by_education.plot(kind="bar", figsize=(8, 5))
plt.title("Income Count by Education Level")
plt.xlabel("Education Level")
plt.ylabel("Number of Employees")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Check income count by age group
income_by_age_group = pd.crosstab(df["age_group"], df["income"])

income_by_age_group.plot(kind="bar", figsize=(8, 5))
plt.title("Income Count by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Employees")
plt.xticks(rotation=30)
plt.show()

## 6. Target selection and feature selection

In [ ]:
# Select target column
# income is the classification target
y = df["income"]

# Drop target column from features
X = df.drop("income", axis=1)

# Drop some raw columns that are already represented by engineered columns
# This keeps the feature set slightly cleaner for teaching
X = X.drop(columns=["fnlwgt", "capital-gain", "capital-loss"], errors="ignore")

print("Final feature columns:")
print(X.columns.tolist())
print("
Feature shape:", X.shape)

## 7. Identify numeric and categorical features

In [ ]:
# Numeric columns will be scaled
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Categorical columns will be one-hot encoded
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("
Categorical features:")
print(categorical_features)

## 8. Train-test split

In [ ]:
# Split the data into training and testing data
# stratify=y ensures both classes are represented properly in train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## 9. Build preprocessing pipeline

In [ ]:
# Scale numeric features
numeric_transformer = StandardScaler()

# Encode categorical features
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created")

## 10. Build Logistic Regression model

In [ ]:
# Create final pipeline with preprocessing and Logistic Regression
# class_weight='balanced' helps when target classes are imbalanced
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

print(model)

## 11. Model training

In [ ]:
# Train the model
model.fit(X_train, y_train)

print("Model training completed")

## 12. Model evaluation

In [ ]:
# Predict class labels on test data
y_pred = model.predict(X_test)

# Predict probabilities for ROC-AUC calculation
# We take probability of the positive class '>50K'
y_prob = model.predict_proba(X_test)[:, list(model.classes_).index(">50K")]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=">50K")
recall = recall_score(y_test, y_pred, pos_label=">50K")
f1 = f1_score(y_test, y_pred, pos_label=">50K")
roc_auc = roc_auc_score((y_test == ">50K").astype(int), y_prob)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC-AUC:", roc_auc)

print("
Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Display confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Display ROC curve
RocCurveDisplay.from_predictions((y_test == ">50K").astype(int), y_prob)
plt.title("ROC Curve")
plt.show()

## 13. Predict for a new employee

In [ ]:
# Create a sample new employee record
# It must contain the same feature columns used during training
new_employee = pd.DataFrame([{
    "age": 36,
    "workclass": "Private",
    "education": "Bachelors",
    "educational-num": 13,
    "marital-status": "Married-civ-spouse",
    "occupation": "Exec-managerial",
    "relationship": "Husband",
    "race": "White",
    "gender": "Male",
    "hours-per-week": 42,
    "native-country": "United-States",
    "workclass_missing": 0,
    "occupation_missing": 0,
    "native_country_missing": 0,
    "age_group": "Adult",
    "education_level": "Graduate_Level",
    "is_married": 1,
    "is_private_workclass": 1,
    "is_us_native": 1,
    "has_capital_gain": 0,
    "has_capital_loss": 0,
    "capital_net": 0,
    "log_fnlwgt": np.log1p(200000),
    "log_capital_gain": np.log1p(0),
    "log_capital_loss": np.log1p(0),
    "age_x_education": 36 * 13,
    "work_intensity": "Standard_Hours"
}])

# Reorder columns to match training features
new_employee = new_employee[X.columns]

# Predict class and probability
prediction = model.predict(new_employee)
probability = model.predict_proba(new_employee)

print("Predicted income:", prediction[0])
print("Class labels:", model.classes_)
print("Prediction probabilities:", probability)

## 14. Save the final model

In [ ]:
# Save final trained model using pickle
with open("final_logistic_regression_feature_engineering_employee.pkl", "wb") as file:
    pickle.dump(model, file)

print("Model saved as final_logistic_regression_feature_engineering_employee.pkl")